# Assignment: Build a Chatbot using Hugging Face Transformers

## 1. Install Required Libraries

In [1]:
!pip install transformers torch

## 2. Import Libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

## 3. Load Chatbot Model

In [3]:
# Load pretrained model and tokenizer
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded successfully!


## 4. Basic Chatbot Loop (First Version)

In [8]:

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end.")

# Initialize chat history
chat_history_ids = None

while True:
    # Take input
    user_input = input("You: ")
    
    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # Improve responses for questions
    if any(word in user_input.lower() for word in ["what", "who", "where", "when", "why", "how"]):
        user_input = "Answer this question: " + user_input

    # Encode input
    new_input = user_input + tokenizer.eos_token
    inputs = tokenizer(new_input, return_tensors='pt')

    new_input_ids = inputs['input_ids']

    # Append chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Create proper attention mask
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=200,   # reduced for better speed
        pad_token_id=tokenizer.eos_token_id,
        
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        no_repeat_ngram_size=2,
        repetition_penalty=1.2
    )

    # Decode only new response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # Print response
    print("Chatbot:", bot_response.strip())

Chatbot: Hello! I am your AI assistant. How can I help you today?


User:  Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hey there, how are you?


User:  hello


Chatbot: Hey, how is your day going?


User:  Who created Python?


Chatbot: What are you asking?


User:  exit


Chatbot: Goodbye! Ending the conversation.


# Chatbot using Hugging Face Transformers

## Objective
To build a conversational chatbot using a pre-trained transformer model (DialoGPT) that can interact with users and generate meaningful responses.

## Tools & Technologies
- Python
- Hugging Face Transformers
- PyTorch
- Jupyter Notebook